# 02 — Federated Training

This notebook runs the single-machine FL simulation using the Flower framework.

**Prerequisites**: Run `01_data_distribution.ipynb` first to create client data splits.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import pickle

import flwr as fl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from flwr.common import ndarrays_to_parameters
from flwr.server import ServerConfig

from src.config import (
    N_CLIENTS, NUM_ROUNDS, LOCAL_EPOCHS, FEATURE_COLUMNS,
    MODELS_DIR, PLOTS_DIR
)
from src.data_utils import load_client_data, prepare_client_tensors, get_global_scaler
from src.models import initialise_model, get_model_parameters
from src.client import DiabetesClient
from src.strategies import LoggingFedAvg
from src.metrics import save_metrics

logging.basicConfig(level=logging.WARNING)  # suppress verbose Flower logs in notebook
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Load Client Data

In [ ]:
client_dfs = {i: load_client_data(i) for i in range(N_CLIENTS)}
scaler = get_global_scaler(client_dfs)
n_features = len(FEATURE_COLUMNS)

for cid, df in client_dfs.items():
    print(f'Client {cid}: {len(df):,} rows, positive_rate={df["Diabetes_binary"].mean():.3f}')

## 2. Build Strategy & client_fn

In [ ]:
global_model = initialise_model(n_features)
initial_params = ndarrays_to_parameters(get_model_parameters(global_model))

strategy = LoggingFedAvg(
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=N_CLIENTS,
    min_evaluate_clients=N_CLIENTS,
    min_available_clients=N_CLIENTS,
    initial_parameters=initial_params,
)

def client_fn(cid: str) -> DiabetesClient:
    df = client_dfs[int(cid)]
    X_train, y_train, X_test, y_test, _ = prepare_client_tensors(df, scaler=scaler)
    model = initialise_model(n_features)
    return DiabetesClient(
        model=model, X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test, client_id=cid, local_epochs=LOCAL_EPOCHS
    )

print(f'Strategy: FedAvg | Rounds: {NUM_ROUNDS} | Clients: {N_CLIENTS}')

## 3. Run Simulation

In [ ]:
history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=N_CLIENTS,
    config=ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy,
    client_resources={'num_cpus': 1, 'num_gpus': 0},
)
print('Simulation complete!')

## 4. Inspect Per-Round Metrics

In [ ]:
import pandas as pd
metrics_df = pd.DataFrame(strategy.metrics_history)
print(metrics_df[['round','loss','accuracy','f1','roc_auc']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(metrics_df['round'], metrics_df['accuracy'], marker='o', color='steelblue')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Global Accuracy per Round')
axes[0].grid(True, alpha=0.3)

axes[1].plot(metrics_df['round'], metrics_df['loss'], marker='o', color='firebrick')
axes[1].set_xlabel('Round')
axes[1].set_ylabel('Log Loss')
axes[1].set_title('Global Loss per Round')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Federated Learning Convergence', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/convergence_curves.png', bbox_inches='tight')
plt.show()

## 5. Save Results

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

save_metrics(
    {'fl_metrics_history': strategy.metrics_history},
    MODELS_DIR / 'fl_metrics_history.json'
)

# Save global model
from src.models import set_model_parameters
from flwr.common import parameters_to_ndarrays

if strategy.parameters:
    arrays = parameters_to_ndarrays(strategy.parameters)
    final_model = initialise_model(n_features)
    set_model_parameters(final_model, arrays)
    with open(MODELS_DIR / 'global_model.pkl', 'wb') as f:
        pickle.dump(final_model, f)
    print('Global model saved.')

print('All results saved.')